In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd().parents[0]
os.chdir(repo_root)

print(Path.cwd())

In [ ]:
repo_root

In [ ]:
from scripts.data_loader import load_locality_data
import scripts.weed_simulation_tools as wst

In [ ]:
localities = load_locality_data()

metro = localities[
    localities["postcode"].astype(int).between(6000,6199)
].copy()

metro['weed_index'] = wst.weed_index_initialize(density=metro['population_density_km2'],
                                            pressure_coef=0.05,
                                            w_min=0.05,
                                            w_max=1)

In [ ]:
type(localities)

In [ ]:
type(metro)

In [ ]:
metro.columns

In [ ]:
metro.shape

In [ ]:
metro["population_density_km2"].hist()

In [ ]:
metro["weed_index"].hist()

In [ ]:
metro["population_density_km2"].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
)

In [ ]:
metro["weed_index"].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
)

In [ ]:
for coefficient in [0.05, 0.005, 0.001, 0.000783, 0.0005]:
    column_name = f"weed_index_k_{coefficient}"

    metro[column_name] = wst.weed_index_initialize(
        density=metro["population_density_km2"],
        pressure_coef=coefficient,
        w_min=0.05,
        w_max=1.0,
    )

    print(f"\npressure_coef = {coefficient}")
    print(
        metro[column_name].describe(
            percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
        )
    )

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))

metro.plot(
    column="weed_index",
    cmap="YlGn",
    legend=True,
    ax=ax,
    edgecolor="none"
)

ax.set_title("Initial Weed Density Index")
ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
len(metro)

In [ ]:
import scripts.weed_simulation_tools as wst
import numpy as np

In [ ]:
weed_grid, valid_mask, transform = wst.polygon_to_grid(
    metro,
    cell_size=1000
)

print(weed_grid.shape)
print(np.nanmin(weed_grid))
print(np.nanmax(weed_grid))

In [ ]:
weed_grid

In [ ]:
import matplotlib.pyplot as plt

cmap = plt.cm.YlGn.copy()
cmap.set_bad("lightgray")

plt.figure(figsize=(10, 10))

plt.imshow(
    weed_grid,
    cmap=cmap,
    vmin=0,
    vmax=1
)

plt.colorbar(label="Weed Index")

plt.title("Initial Weed Index Grid")
plt.axis("off")

plt.show()

In [ ]:
import importlib
import scripts.weed_ca

importlib.reload(scripts.weed_ca)

weedca = scripts.weed_ca.WeedCA.polygon_to_weedca(metro, 
                                                  pressure_coef=0.05,
                                                  cell_size=500,
                                                  growth_rate=0.02,
                                                  diffusion_rate=0.01)

modes = ['growth', 'diffusion']
weedca.animate(frames=108, fps=20, modes=modes)

In [ ]:
def simple_removal(weed_grid, mask):
    weed_grid[~mask] = 0
    diff = weed_grid * 0.001
    cost = np.sum(diff * 200)

    return diff, cost

weedca.register_policy(simple_removal)

In [ ]:
modes = ['growth', 'diffusion', 'removal']
weedca.animate(frames=108, fps=20, modes=modes)